# The recession ended; the damage did not

Article #25 companion notebook. The historical data are already frozen in data/raw. The observational comparisons do **not** identify the effects of TARP, the Fed, stimulus, or regulation; the final calculation is entirely hypothetical. See article.md and data/sources.json for definitions and archival hashes.

In [ ]:
from datetime import date
from analysis import equity_constrained_sales, milestones, racial_outcomes, read_wealth
from prepare_data import build_panel, load_sources, prepared_outputs
panel = build_panel()
sources = load_sources()
assert len(panel) == 13
print('Annual records:', len(panel), ' | first and last:', panel[0]['year'], panel[-1]['year'])

## Four clocks, four benchmarks

Late-2007 historical levels are used as benchmarks, not as projected no-crisis trends. First-recovery dates depend on their chosen denominators and observation frequencies.

In [ ]:
for m in milestones():
    print(f"{m['indicator']:30}  {m['baseline_date']} → {m['recovery_date']}")
assert {m['indicator']: m['recovery_date'] for m in milestones()}['Payroll jobs'] == '2014-05-01'
for row in (panel[0], panel[2], panel[-1]):
    print(row['year'], 'unemployment', round(row['unemployment_pct'], 2), 'median income', round(row['real_median_household_income_2025_usd']))

## Repeated-cross-section wealth, not individual trajectories

The five SCF waves are different survey samples. Values are thousands of 2022 dollars. The ratios are comparisons of group medians, not group mean wealth or confidence intervals.

In [ ]:
wealth = read_wealth()
for year, row in wealth.items():
    black_ratio = 100 * row['Black'] / row['White']
    hispanic_ratio = 100 * row['Hispanic'] / row['White']
    print(year, f'White ${row["White"]:.2f}k', f'Black/White {black_ratio:.1f}%', f'Hispanic/White {hispanic_ratio:.1f}%')
assert round(racial_outcomes()[-1]['black_white_median_wealth_ratio_pct'], 1) == 12.8

## Hypothetical bank-capital identity

For an invented bank with 100 initial asset units, starting equity share k0, markdown ℓ, and a required book equity-to-assets ratio k*, further asset sales as a share of initial assets are max(0, 1−ℓ−(k0−ℓ)/k*). These are *additional* to the markdown and presume sales at unchanged book value and debt repayment. This does not model actual bank balance sheets or behavior.

In [ ]:
for initial in (0.08, 0.10, 0.12):
    sale = equity_constrained_sales(0.02, initial, 0.08)
    print(f'Starting equity {initial:.0%}; two-percent asset loss → further sales {sale:.1%} of original assets')
assert abs(equity_constrained_sales(0.02, 0.08, 0.08) - 0.23) < 1e-12
print('Frozen derived tables audited:', all(p.read_text(encoding='utf-8') == expected for p, expected in prepared_outputs().items()))

## Figures and full audit

Run python analysis.py to regenerate eight PNG publication figures and three result tables, then python -m unittest discover -s tests -v for publication and source-integrity checks. The figures have deliberately different units; do not treat proximity or parallel movements as causal estimates.